### Gráficos para Artigo SBRC 2026 - LSTM Recursiva
Possibilidades de gráficos para submissão de artigo da SBRC 2026 com o título - Predição Iterativa de Vazão de Rede Utilizando Uma Abordagem de Janelas Deslizantes combinada com Imputação de Dados Dinâmica

1. Análise de melhores links para mostrar nos resultados  
Considerando último treinamento, onde acabou que o processo da LSTM recursiva morreu  
Diminuir a amostra para poder justifiá-la melhor na sessão de experimentos 

In [18]:
from __future__ import annotations

import argparse
import csv
import json
from dataclasses import dataclass
from pathlib import Path 

# Usado para avaliar se os links a serem escolhidos vão estar espalhados no mapa
# Para justificar a escolha dos links
REGION_BY_STATE = {
    "ac": "Norte",
    "al": "Nordeste",
    "ap": "Norte",
    "am": "Norte",
    "ba": "Nordeste",
    "ce": "Nordeste",
    "df": "Centro-Oeste",
    "es": "Sudeste",
    "go": "Centro-Oeste",
    "ma": "Nordeste",
    "mt": "Centro-Oeste",
    "ms": "Centro-Oeste",
    "mg": "Sudeste",
    "pa": "Norte",
    "pb": "Nordeste",
    "pr": "Sul",
    "pe": "Nordeste",
    "pi": "Nordeste",
    "rj": "Sudeste",
    "rn": "Nordeste",
    "rs": "Sul",
    "ro": "Norte",
    "rr": "Norte",
    "sc": "Sul",
    "sp": "Sudeste",
    "se": "Nordeste",
    "to": "Norte",
}

ALL_REGIONS = {"Norte", "Nordeste", "Centro-Oeste", "Sudeste", "Sul"}
REQUIRED_MODELS = ("Proposal", "GRU", "LSTM", "SARIMA")

# Classificar pelas métricas
@dataclass(frozen=True)
class LinkMetrics:
    link: str
    state_a: str
    state_b: str
    region_a: str
    region_b: str
    cross_region: bool
    nrmse_proposal: float
    nrmse_gru: float
    nrmse_lstm: float
    nrmse_sarima: float

    @property
    def best_baseline(self):
        return min(self.nrmse_gru, self.nrmse_lstm, self.nrmse_sarima)

    @property
    def mean_baseline(self):
        return (self.nrmse_gru + self.nrmse_lstm + self.nrmse_sarima) / 3.0

    @property
    def proposal_advantage_vs_best(self):
        return self.best_baseline - self.nrmse_proposal

    @property
    def proposal_advantage_vs_mean(self):
        return self.mean_baseline - self.nrmse_proposal

    @property
    def proposal_rank(self):
        scores = [
            ("Proposal", self.nrmse_proposal),
            ("GRU", self.nrmse_gru),
            ("LSTM", self.nrmse_lstm),
            ("SARIMA", self.nrmse_sarima),
        ]
        scores_sorted = sorted(scores, key=lambda item: item[1])
        for idx, (name, _) in enumerate(scores_sorted, start=1):
            if name == "Proposal":
                return idx
        return len(scores_sorted)


@dataclass
class SelectionResult:
    selected: list
    missing_regions: set
    cross_region_count: int
    cross_region_share: float
    min_cross_region: int
    threshold_used: float

# Parsers e loadings
def parse_link(link):
    parts = link.lower().split("-")
    if len(parts) != 2:
        raise ValueError(f"Link inválido: {link}")
    return parts[0], parts[1]


def get_region(state: str) -> str:
    if state not in REGION_BY_STATE:
        raise KeyError(f"UF desconhecida no link: {state}")
    return REGION_BY_STATE[state]


def load_results(csv_path):
    results = {}
    with csv_path.open(newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            link = row["Link"].strip().lower()
            model = row["Model"].strip()
            try:
                nrmse = float(row["NRMSE"])
            except (ValueError, TypeError):
                continue
            results.setdefault(link, {})[model] = nrmse
    return results


# Comparativos
def build_metrics(records):
    metrics = []
    for link, model_data in records.items():
        if not all(model in model_data for model in REQUIRED_MODELS):
            continue
        state_a, state_b = parse_link(link)
        region_a = get_region(state_a)
        region_b = get_region(state_b)
        metrics.append(
            LinkMetrics(
                link=link,
                state_a=state_a,
                state_b=state_b,
                region_a=region_a,
                region_b=region_b,
                cross_region=region_a != region_b,
                nrmse_proposal=model_data["Proposal"],
                nrmse_gru=model_data["GRU"],
                nrmse_lstm=model_data["LSTM"],
                nrmse_sarima=model_data["SARIMA"],
            )
        )
    return metrics


def regions_in_links(links):
    regions = set()
    for link in links:
        regions.add(link.region_a)
        regions.add(link.region_b)
    return regions


def count_cross_region(links):
    return sum(1 for link in links if link.cross_region)


def select_samples(
    metrics,
    min_advantage,
    max_regression,
    min_cross_region
):
    threshold = min_advantage
    selected = [link for link in metrics if link.proposal_advantage_vs_best >= threshold]
    print(f"selected: {selected}")
    missing_regions = ALL_REGIONS - regions_in_links(selected)
    cross_region_count = count_cross_region(selected)

    if (missing_regions or cross_region_count < min_cross_region) and max_regression > 0:
        threshold = -max_regression
        selected = [link for link in metrics if link.proposal_advantage_vs_best >= threshold]
        missing_regions = ALL_REGIONS - regions_in_links(selected)
        cross_region_count = count_cross_region(selected)

    cross_region_share = (
        cross_region_count / len(selected) if selected else 0.0
    )
    return SelectionResult(
        selected=sorted(selected, key=lambda item: item.link),
        missing_regions=missing_regions,
        cross_region_count=cross_region_count,
        cross_region_share=cross_region_share,
        min_cross_region=min_cross_region,
        threshold_used=threshold,
    )


# def write_selection_csv(selection, output_path):
#     output_path.parent.mkdir(parents=True, exist_ok=True)
#     with output_path.open("w", newline="", encoding="utf-8") as f:
#         writer = csv.writer(f)
#         writer.writerow(
#             [
#                 "Link",
#                 "State_A",
#                 "State_B",
#                 "Region_A",
#                 "Region_B",
#                 "Cross_Region",
#                 "NRMSE_Proposal",
#                 "NRMSE_GRU",
#                 "NRMSE_LSTM",
#                 "NRMSE_SARIMA",
#                 "Proposal_Advantage_vs_Best",
#                 "Proposal_Advantage_vs_Mean",
#                 "Proposal_Rank",
#             ]
#         )
#         for link in selection:
#             writer.writerow(
#                 [
#                     link.link,
#                     link.state_a,
#                     link.state_b,
#                     link.region_a,
#                     link.region_b,
#                     str(link.cross_region).lower(),
#                     f"{link.nrmse_proposal:.6f}",
#                     f"{link.nrmse_gru:.6f}",
#                     f"{link.nrmse_lstm:.6f}",
#                     f"{link.nrmse_sarima:.6f}",
#                     f"{link.proposal_advantage_vs_best:.6f}",
#                     f"{link.proposal_advantage_vs_mean:.6f}",
#                     str(link.proposal_rank),
#                 ]
#             )


# def write_summary_json(result, output_path) -> None:
#     output_path.parent.mkdir(parents=True, exist_ok=True)
#     payload = {
#         "selected_links": len(result.selected),
#         "missing_regions": sorted(result.missing_regions),
#         "cross_region_links": result.cross_region_count,
#         "cross_region_share": round(result.cross_region_share, 4),
#         "min_cross_region_required": result.min_cross_region,
#         "threshold_used": result.threshold_used,
#         "regions_present": sorted(regions_in_links(result.selected)),
#     }
#     with output_path.open("w", encoding="utf-8") as f:
#         json.dump(payload, f, ensure_ascii=False, indent=2)


In [19]:
from pathlib import Path

# Entradas / saídas
INPUT_CSV  = Path("final_comparison_results.csv")
OUTPUT_CSV = Path("selection_samples.csv")
SUMMARY_JSON = Path("selection_summary.json")

# Critérios de seleção (equivalentes aos argumentos)
MIN_ADVANTAGE = 0.0       # --min-advantage
MAX_REGRESSION = 0.1      # --max-regression (positivo permite proposta levemente pior)
MIN_CROSS_REGION = 8      # --min-cross-region

In [20]:
# Carrega o CSV consolidado
records = load_results(INPUT_CSV)

# Constrói métricas por link (filtra só links que têm Proposal/GRU/LSTM/SARIMA)
metrics = build_metrics(records)

if not metrics:
    raise SystemExit("Nenhum link completo (Proposal/GRU/LSTM/SARIMA) encontrado.")

# Seleciona links com foco na vantagem da Proposal
selection = select_samples(
    metrics,
    min_advantage=MIN_ADVANTAGE,
    max_regression=MAX_REGRESSION,
    min_cross_region=MIN_CROSS_REGION,
)

# (Opcional) Salva artefatos
# write_selection_csv(selection.selected, OUTPUT_CSV)
# write_summary_json(selection, SUMMARY_JSON)


selected: [LinkMetrics(link='ma-pb', state_a='ma', state_b='pb', region_a='Nordeste', region_b='Nordeste', cross_region=False, nrmse_proposal=0.006095671056292221, nrmse_gru=0.00752358093226591, nrmse_lstm=0.00764631095701348, nrmse_sarima=0.007666550715907363)]


In [23]:
print("=== Seleção de Amostras SBRC 2026 ===")
print(f"Total de links completos: {len(metrics)}")
print(f"Links selecionados: {len(selection.selected)}")
print(f"Limiar aplicado (advantage vs best): {selection.threshold_used:.3f}")
print(f"Links inter-regionais: {selection.cross_region_count}")
print(f"Cobertura regional: {', '.join(sorted(regions_in_links(selection.selected)))}")

if selection.missing_regions:
    print(
        "Aviso: regiões ausentes na seleção -> "
        + ", ".join(sorted(selection.missing_regions))
    )

# print(f"CSV salvo em: {OUTPUT_CSV}")
# print(f"Resumo salvo em: {SUMMARY_JSON}")

print(f"Seleção: {selection}")

=== Seleção de Amostras SBRC 2026 ===
Total de links completos: 29
Links selecionados: 5
Limiar aplicado (advantage vs best): -0.100
Links inter-regionais: 3
Cobertura regional: Centro-Oeste, Nordeste, Norte, Sudeste, Sul
Seleção: SelectionResult(selected=[LinkMetrics(link='ce-sc', state_a='ce', state_b='sc', region_a='Nordeste', region_b='Sul', cross_region=True, nrmse_proposal=0.0870696865286412, nrmse_gru=0.028121189817868854, nrmse_lstm=0.028025355045275247, nrmse_sarima=0.027308182215988352), LinkMetrics(link='ce-sp', state_a='ce', state_b='sp', region_a='Nordeste', region_b='Sudeste', cross_region=True, nrmse_proposal=0.09236147743697007, nrmse_gru=0.025364970375387947, nrmse_lstm=0.025819847736201, nrmse_sarima=0.026978570869676973), LinkMetrics(link='ma-pb', state_a='ma', state_b='pb', region_a='Nordeste', region_b='Nordeste', cross_region=False, nrmse_proposal=0.006095671056292221, nrmse_gru=0.00752358093226591, nrmse_lstm=0.00764631095701348, nrmse_sarima=0.007666550715907363

In [24]:
links = [lm.link for lm in selection.selected]
print(f"Links selecionados: {links}")

Links selecionados: ['ce-sc', 'ce-sp', 'ma-pb', 'ma-pi', 'to-go']


2. Gráficos de cada métrica de comparação: RMSE, MAE, Fuzzy, etc

2.1. Tabela de cada métrica de comparação 

3. Gráficos comparativos gerais: Heatmap e "count wins"